# Clase 4 · 00 · Preflight: streaming en Databricks

Hasta ahora procesamos datos en **batch**: un archivo llega, lo procesamos completo y terminamos. En **streaming** los datos no terminan nunca de llegar y queremos reaccionar a medida que aparecen.

| Notebook | Tema de la teoría | Qué vas a ver |
|---|---|---|
| `01_log_y_offsets` | Pub/Sub, log, offsets, fan-out, replay | Una tabla Delta funcionando como un *topic* de Kafka, con dos consumidores independientes |
| `02_ingesta_y_enriquecimiento` | Ingesta incremental, calidad, join stream-tabla | Cinco llegadas de archivos procesadas con Auto Loader y enriquecidas con clientes y productos |
| `03_tiempo_y_ventanas` | Event time, watermark, ventanas, datos tardíos | Tumbling, hopping y session; un evento tardío aceptado y otro descartado; append vs. update |
| `04_garantias_y_fallas` | At-least-once, exactly-once, idempotencia | Duplicados, reejecuciones, pérdida de checkpoint y escritura idempotente con `MERGE` |
| `05_cdc` | Change Data Capture, estado y stream | Los cambios de una tabla leídos como stream y la tabla reconstruida desde ese log |

**Los notebooks son demostrativos:** no hay que modificar código. Ejecutalos en orden, celda por celda, leé el texto y mirá los resultados. Las 20 preguntas de la entrega están en el `README.md`.

En Databricks Free Edition (serverless) los streams se ejecutan con `trigger(availableNow=True)`: la consulta procesa todo lo pendiente y **termina**, y el checkpoint recuerda hasta dónde llegó. Es exactamente lo que hace un stream continuo, pero en pasos que podemos observar.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_rules

In [ ]:
%run ../common/streaming_lab

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
spark.conf.set("spark.sql.session.timeZone", "UTC")
customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
for table in [customers, products]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá la clase 2."
T = lambda name: qualified_table(config, name)

In [ ]:
counts = {name: spark.table(table).count() for name, table in [("silver_customers", customers), ("silver_products", products)]}
assert counts["silver_customers"] == config.rows["customers"], "student_id o scale no coinciden con las clases anteriores"
assert counts["silver_products"] == config.rows["products"], "student_id o scale no coinciden con las clases anteriores"
display(spark.createDataFrame([(k, v) for k, v in counts.items()], ["tabla", "filas"]))
print(f"Laboratorio de streaming: {lab_path(config)}")

## Los eventos del experimento

Un productor ficticio envía compras en **cinco llegadas** (`stream_001` … `stream_005`). Todas ocurren el 12/03/2026 entre las 12:00 y las 12:45 (UTC). Este es el bloque base. Con escala `small` se repite 20 veces con otros IDs y clientes.

In [ ]:
rows = []
for batch in ["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"]:
    for r in build_stream_rows(config, batch):
        if r["event_id"].endswith("_000"):
            rows.append({"llegada": batch, "event_id": r["event_id"], "hora del evento": r["event_ts"][11:16],
                         "amount": r["amount"], "canal": r["payment_channel"], "tipo": r["event_type"]})
display(pd.DataFrame(rows))

Fijate en algunos eventos especiales que vamos a seguir:

- `e002` llega **dos veces** (en las llegadas 1 y 2): un duplicado, como cuando un productor reintenta un envío.
- `late_ok` ocurrió a las 12:03 pero llega recién en la llegada 2: **tardío, pero no tanto**.
- `late_bad` ocurrió a las 12:02 y llega en la llegada 4, cuando ya pasaron más de 20 minutos: **demasiado tardío**.
- `bad_amount` y `bad_customer` tienen errores de calidad.
- `clock` (llegada 5) no es una compra: es un evento de control que hace avanzar el tiempo para cerrar las últimas ventanas.